# Data Science Project: Planning Stage (Individual)

*Isabel (Yibao) Zhang*

***GITHUB REPO FOR THIS PROJECT WITH RESPECTIVE COMMITS:*** https://github.com/IsabelZ27/Dsci-100-Project-Individual/tree/main

In [39]:
library(tidyverse)
library(repr)
library(tidymodels)
library(cowplot)
library(lubridate)
options(repr.matrix.max.rows = 6)

### Part 1: Data Description

In [40]:
# Load the used data - sessions.csv
sessions <- read_csv("sessions.csv")
sessions
# Find a example when end_time is NA
example_NA <- sessions |> slice(681)
example_NA
# Summarize for original_start_time	and original_end_time
summary(sessions$original_start_time)
summary(sessions$original_end_time)

Rows: 1535 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): hashedEmail, start_time, end_time
dbl (2): original_start_time, original_end_time

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


hashedEmail,start_time,end_time,original_start_time,original_end_time
<chr>,<chr>,<chr>,<dbl>,<dbl>
bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431d8aa0c4bf95ccee6bf,30/06/2024 18:12,30/06/2024 18:24,1.71977e+12,1.71977e+12
36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f575d4acc9cf487c4686,17/06/2024 23:33,17/06/2024 23:46,1.71867e+12,1.71867e+12
f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3398304c7ae42581fdc,25/07/2024 17:34,25/07/2024 17:57,1.72193e+12,1.72193e+12
⋮,⋮,⋮,⋮,⋮
fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33cbb5e894a3867ca44d,28/07/2024 15:36,28/07/2024 15:57,1.72218e+12,1.72218e+12
fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33cbb5e894a3867ca44d,25/07/2024 06:15,25/07/2024 06:22,1.72189e+12,1.72189e+12
36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f575d4acc9cf487c4686,20/05/2024 02:26,20/05/2024 02:45,1.71617e+12,1.71617e+12


hashedEmail,start_time,end_time,original_start_time,original_end_time
<chr>,<chr>,<chr>,<dbl>,<dbl>
55d24216db39c27e1f17cc43d3127cbf8ed76ada6d098202b53ded319855e2c1,27/08/2024 17:06,NA,1.72478e+12,NA


     Min.   1st Qu.    Median      Mean   3rd Qu.      Max. 
1.712e+12 1.716e+12 1.719e+12 1.719e+12 1.722e+12 1.727e+12 

     Min.   1st Qu.    Median      Mean   3rd Qu.      Max.      NA's 
1.712e+12 1.716e+12 1.719e+12 1.719e+12 1.722e+12 1.727e+12         2 

In this project, we are working with the sessions.csv dataset from the UBC Minecraft research server. The dataset includes individual play sessions of players on the server.<br>
**Summary:**<br>
Observations: There are 1535 rows, which means we have collected 1535 observations in total.<br>
Variables: There are 5 variables: hashedEmail (identifier), start_time (string), end_time (string), original_start_time (numeric variable), and original_end_time (numeric variable).<br>
    - hashedEmail means users' email information.<br>
    - start_time means the time that users start to play Minecraft on the server.<br>
    - end_time means the time that users stop playing Minecraft on the server.<br>
    - original_start_time means the start time after transferring to UNIX time (milliseconds).<br>
    - original_end_time means the end time after transferring to UNIX time (milliseconds). <br>
Summary statistics: We are unable to calculate the mean, median and max/min for hashedEmail, start_time, and end_time because these variables are not numeric variables. According to the calculations above, we can summarize some statistics for original_start_time and original_end_time.<br>
Issues & Potential issues: <br>
    - The original_start_time and original_end_time were rounded to 6 digits, which heavily reduced the data accuracy and usefulness.<br>
    - There were NAs under the variable end_time (likely due to server issues), for example, the 681th row, which messes with our calculations quite a lot, as we do not know when the user got off.<br>
    - Very low player counts generally player counts stayed around the 1-2 player mark for a majority of the data's reach, with the maximum peak only reaching 15, meaning the player count statistic will be much more volatile and harder to predict accurately.<br>


### Part 2: Questions

I chose broad question 3: "We are interested in demand forecasting, namely, what time windows are most likely to have a large number of simultaneous players. This is because we need to ensure that the number of licenses on hand is sufficiently large to accommodate all parallel players with high probability."<br>
My specific question: “Can we predict the number of simultaneous players in any given hour in a day based on historical player activity (start_time and end_time)?”<br>
In my specific question, the response variable is concurrent (numeric count of players in overlapping sessions per hourly bin). The given data provides start_time and end_time for each observation, which can help us summarize and calculate the concurrent.<br>
Meanwhile, the start_time and end_time also provide us with the dates, hour-of-day, weekdays/weekends, etc., which all can be our explanatory variables.<br>
I plan to wrangle my data to get concurrent and sort them based on the timeline. 

### Part 3: Exploratory Data Analysis and Visualization

In [41]:
# standardize the date formats using day_hm
sessions <- read_csv("sessions.csv") |>
            mutate(start_time = dmy_hm(start_time), end_time = dmy_hm(end_time))

Rows: 1535 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): hashedEmail, start_time, end_time
dbl (2): original_start_time, original_end_time

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [42]:
# Identify the NAs to filter
sessions <- sessions |> filter(!is.na(start_time) & !is.na(end_time))

sessions

hashedEmail,start_time,end_time,original_start_time,original_end_time
<chr>,<dttm>,<dttm>,<dbl>,<dbl>
bfce39c89d6549f2bb94d8064d3ce69dc3d7e72b38f431d8aa0c4bf95ccee6bf,2024-06-30 18:12:00,2024-06-30 18:24:00,1.71977e+12,1.71977e+12
36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f575d4acc9cf487c4686,2024-06-17 23:33:00,2024-06-17 23:46:00,1.71867e+12,1.71867e+12
f8f5477f5a2e53616ae37421b1c660b971192bd8ff77e3398304c7ae42581fdc,2024-07-25 17:34:00,2024-07-25 17:57:00,1.72193e+12,1.72193e+12
⋮,⋮,⋮,⋮,⋮
fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33cbb5e894a3867ca44d,2024-07-28 15:36:00,2024-07-28 15:57:00,1.72218e+12,1.72218e+12
fd6563a4e0f6f4273580e5fedbd8dda64990447aea5a33cbb5e894a3867ca44d,2024-07-25 06:15:00,2024-07-25 06:22:00,1.72189e+12,1.72189e+12
36d9cbb4c6bc0c1a6911436d2da0d09ec625e43e6552f575d4acc9cf487c4686,2024-05-20 02:26:00,2024-05-20 02:45:00,1.71617e+12,1.71617e+12


In [43]:
# Find out the concurrent and organize the data
hours <- seq(floor_date(min(sessions$start_time), "hour"),
             ceiling_date(max(sessions$end_time), "hour"),
             by = "hour")
hourly <- tibble(hour = hours, concurrent = sapply(hours, function(h){sum(sessions$start_time < h + hours(1) & sessions$end_time > h)}))
hourly
hourly <- tibble(hour = hours, concurrent = sapply(hours, function(h){sum(sessions$start_time < h + hours(1) & sessions$end_time > h)}))
hourly
hourly <- hourly |> mutate(hour_of_day = hour(hour), day_of_week = wday(hour, label = TRUE))
hourly

hour,concurrent
<dttm>,<int>
2024-04-06 09:00:00,2
2024-04-06 10:00:00,1
2024-04-06 11:00:00,0
⋮,⋮
2024-09-26 06:00:00,1
2024-09-26 07:00:00,1
2024-09-26 08:00:00,0


hour,concurrent
<dttm>,<int>
2024-04-06 09:00:00,2
2024-04-06 10:00:00,1
2024-04-06 11:00:00,0
⋮,⋮
2024-09-26 06:00:00,1
2024-09-26 07:00:00,1
2024-09-26 08:00:00,0


hour,concurrent,hour_of_day,day_of_week
<dttm>,<int>,<int>,<ord>
2024-04-06 09:00:00,2,9,Sat
2024-04-06 10:00:00,1,10,Sat
2024-04-06 11:00:00,0,11,Sat
⋮,⋮,⋮,⋮
2024-09-26 06:00:00,1,6,Thu
2024-09-26 07:00:00,1,7,Thu
2024-09-26 08:00:00,0,8,Thu


### Part 4: Methods and Plan

**k-Nearest Neighbours Regression**<br>
Why appropriate: kNN is flexible and non‑parametric. It can capture non-linear patterns between hour-of-day, historical activity, and concurrent player counts.<br>
Assumptions: Similar inputs yield similar outputs. No distributional assumptions.<br>
Limitations: The model requires feature scaling, may struggle with high-dimensional data, and is computationally expensive for large datasets.
Comparison: I will compare the kNN regression with linear regression to see which model works better to predict in this scenario.

In [44]:
summary(hourly$concurrent)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0000  0.0000  0.0000  0.6688  1.0000 15.0000 

However, after summarizing the concurrent data, I found out that the maximum number of concurrent is 15, and the mean is 0.6688, even smaller than 1. This means that normally there are only 1 or less than 1 players in the server, and predicting the number of simultaneous players in any given hour will be less useful. Therefore, I would consider splitting the data into weekdays/weekends to predict the differences.

***GITHUB REPO FOR THIS PROJECT WITH RESPECTIVE COMMITS:*** https://github.com/IsabelZ27/Dsci-100-Project-Individual/tree/main